# Multi-Layer Perceptron (MLP) from Scratch 🧠

In this final notebook, we implement a **Multi-Layer Perceptron (MLP)**, a basic feed-forward neural network, from the ground up.

## 📖 Theoretical Background

An MLP consists of at least three layers of nodes: an input layer, a hidden layer, and an output layer. 

### 1. Forward Propagation
For each layer $l$:
$$z^{[l]} = a^{[l-1]} W^{[l]} + b^{[l]}$$
$$a^{[l]} = g(z^{[l]})$$
Where $g$ is an activation function (like ReLU or Sigmoid).

### 2. Activation Functions
- **ReLU**: $g(z) = \max(0, z)$
- **Sigmoid**: $g(z) = \frac{1}{1 + e^{-z}}$

### 3. Backpropagation (The Chain Rule)
To train the network, we compute the gradient of the loss function with respect to each weight by applying the chain rule, moving backward from the output layer to the input layer.

### 4. Loss Function (Binary Cross-Entropy)
For binary classification:
$$J = -\frac{1}{m} \sum [y \log(\hat{y}) + (1 - y) \log(1 - \hat{y})]$$

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

np.random.seed(42)

## 🏗️ The Implementation

In [ ]:
class MLP:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.01):
        self.lr = learning_rate
        
        # Initialize weights and biases
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))
        
        self.loss_history = []

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def _sigmoid_derivative(self, a):
        return a * (1 - a)

    def _relu(self, z):
        return np.maximum(0, z)

    def _relu_derivative(self, z):
        return (z > 0).astype(float)

    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self._relu(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self._sigmoid(self.z2)
        return self.a2

    def backward(self, X, y, output):
        m = y.shape[0]
        
        # Error at output layer
        dz2 = output - y
        dW2 = (1 / m) * np.dot(self.a1.T, dz2)
        db2 = (1 / m) * np.sum(dz2, axis=0, keepdims=True)
        
        # Error at hidden layer
        da1 = np.dot(dz2, self.W2.T)
        dz1 = da1 * self._relu_derivative(self.z1)
        dW1 = (1 / m) * np.dot(X.T, dz1)
        db1 = (1 / m) * np.sum(dz1, axis=0, keepdims=True)
        
        # Update parameters
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

    def fit(self, X, y, epochs=10000):
        y = y.reshape(-1, 1)
        for epoch in range(epochs):
            # Forward pass
            output = self.forward(X)
            
            # Compute loss
            loss = -np.mean(y * np.log(output + 1e-15) + (1 - y) * np.log(1 - output + 1e-15))
            self.loss_history.append(loss)
            
            # Backward pass
            self.backward(X, y, output)
            
            if epoch % 1000 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

    def predict(self, X):
        output = self.forward(X)
        return (output > 0.5).astype(int)

## 🧪 Data Generation and Training

We will train the MLP on the non-linear 'Moons' dataset.

In [ ]:
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

# Define and train model
model = MLP(input_size=2, hidden_size=10, output_size=1, learning_rate=0.1)
model.fit(X_train, y_train, epochs=5000)

predictions = model.predict(X_test)
accuracy = np.sum(predictions.flatten() == y_test) / len(y_test)
print(f"MLP Accuracy: {accuracy * 100:.2f}%")

## 📊 Visualization

In [ ]:
plt.figure(figsize=(12, 5))

# Plot Loss History
plt.subplot(1, 2, 1)
plt.plot(model.loss_history)
plt.title("MLP Loss History")
plt.xlabel("Epoch")
plt.ylabel("Log Loss")

# Plot Decision Boundary
plt.subplot(1, 2, 2)
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))
Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3, cmap="RdBu")
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu", edgecolors="k")
plt.title("MLP Decision Boundary")
plt.show()